# wardowl: an enterprise RAG assistant, at scale, on Kaggle GPUs

The acceptance run an ML platform team would do before production, with `acceptance/kaggle/enterprise_rag.py` from the [wardowl repository](https://github.com/muzakkirhussain011/wardowl). It builds about 107,000 chunks for 20 tenants: 49,920 HR, finance, engineering, legal and board documents (the templates of `acceptance/enterprise.py`, each non-public one with its own canary), one notice per tenant carrying a prompt injection, and the BEIR FiQA corpus, which every tenant shares. It embeds them with BAAI/bge-base-en-v1.5 on a GPU and loads them into FAISS, Chroma, pgvector, Qdrant, Elasticsearch, OpenSearch, Weaviate, Redis and LanceDB, one store at a time, through wardowl's own backends. Permissions are `AllOf(TenantAuthorizer(), MetadataACLAuthorizer())`; on pgvector they also run through Open Policy Agent and OpenFGA behind `CachingAuthorizer`.

About 190 people ask about 1,100 questions, red-team prompts among them, and Qwen2.5-3B-Instruct answers each one from the five chunks wardowl permits. A canary of a document the person may not read, in any retrieved chunk, prompt or answer, fails the run. Answer accuracy, recall@5 and @10 against exact search, latency percentiles and load time and memory are reported.

**Notebook settings:** Accelerator **GPU T4 x2**, **Internet on**. No input is needed; add the output of an earlier run as input to reuse its embeddings and answers. A full run takes about three to five hours (Kaggle stops at twelve) and writes `results.md`, `results.json` and `answers.jsonl` to `/kaggle/working`. `SMOKE = True` checks the setup in about twenty minutes.

In [ ]:
WARDOWL_VERSION = "2.0.*"  # the release under test, from PyPI: "2.0.*" is the newest 2.0.x, or pin one ("2.0.0")
REF = "acceptance/real-world"  # branch, tag or commit whose acceptance/ and benchmarks/ scripts run; pin a commit
SMOKE = False  # True: a short run (small corpus, FAISS and pgvector, Qwen2.5-0.5B) that checks the setup
VARIANTS = 208  # documents per template per tenant: 20 tenants x 12 templates x 208 = 49,920
KB_LIMIT = 0  # FiQA passages every tenant shares (0: all, about 57,600)
STORES = ("faiss", "chroma", "pgvector", "qdrant", "elasticsearch", "opensearch", "weaviate", "redis", "lancedb")
REMOTE_AUTHORIZERS = ("opa", "openfga")  # run on pgvector, after the in-process policy
EMBED_MODEL = "BAAI/bge-base-en-v1.5"
LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
SRC = "/tmp/wardowl-src"  # the clone; only its acceptance/ folders go on sys.path, so the PyPI release runs

In [ ]:
!rm -rf {SRC} && git clone -q https://github.com/muzakkirhussain011/wardowl.git {SRC}
!git -C {SRC} checkout -q {REF} && git -C {SRC} log -1 --format="%H %cs %s"
EXTRAS = "pgvector,chroma,elasticsearch,opensearch,qdrant,weaviate,redis,lancedb,faiss"
!pip install -q "wardowl[{EXTRAS}]=={WARDOWL_VERSION}" "sentence-transformers>=3" "datasets>=2.19" psutil

In [ ]:
import shutil
from pathlib import Path

# Embeddings and answers an earlier run saved (its output added as input) are reused when they match.
cache = Path("/kaggle/working/cache")
cache.mkdir(parents=True, exist_ok=True)
for saved in [
    *Path("/kaggle/input").glob("**/cache/chunks-*.npy"),
    *Path("/kaggle/input").glob("**/cache/answers-*.json"),
]:
    shutil.copy(saved, cache)
    print(f"reusing {saved}")

In [ ]:
# PostgreSQL with pgvector, built from source: Kaggle has no Docker. The other stores, Open Policy Agent
# and OpenFGA are started from their release builds, and stopped again, by the store cells below.
!PG_TUNE=1 bash {SRC}/benchmarks/kaggle_setup.sh

In [ ]:
import sys

sys.path.insert(0, f"{SRC}/acceptance/kaggle")
import enterprise_rag as er

if SMOKE:
    config = er.Config.smoke()
else:
    config = er.Config(
        stores=STORES,
        remote_authorizers=REMOTE_AUTHORIZERS,
        variants=VARIANTS,
        kb_limit=KB_LIMIT,
        embed_model=EMBED_MODEL,
        llm_model=LLM_MODEL,
    )
config.wardowl_version, config.repo = WARDOWL_VERSION, SRC
run = er.Run(config)
run.environment_check()  # versions, GPUs, and that wardowl is the PyPI release

In [ ]:
run.build_corpus()  # 49,920 company documents, 20 injected notices, the FiQA corpus, 1,100 questions

In [ ]:
run.embed()  # BAAI/bge-base-en-v1.5 on the second GPU, cached in /kaggle/working/cache

In [ ]:
run.baseline()  # exact top 10 over what each person may read: a matrix multiply on the GPU

In [ ]:
run.load_llm()  # Qwen2.5-3B-Instruct in fp16 on the first GPU, greedy

## One store per cell

Each cell starts the store, loads every chunk through wardowl's backend, retrieves k=5 and k=10 for every question, has the assistant answer from the k=5 chunks, checks every chunk, prompt and answer against the oracle, and stops the store. A failure is recorded as a failed check and the next cell still runs.

In [ ]:
run.store("faiss")  # exact search, in process: its recall must match brute force

In [ ]:
run.store("chroma")  # embedded, persistent

In [ ]:
run.store("pgvector")  # then the same table with Open Policy Agent and with OpenFGA, behind CachingAuthorizer

In [ ]:
run.store("qdrant")

In [ ]:
run.store("elasticsearch")  # int8_hnsw, num_candidates 500

In [ ]:
run.store("opensearch")

In [ ]:
run.store("weaviate")

In [ ]:
run.store("redis")  # Redis 8 with its Query Engine

In [ ]:
run.store("lancedb")  # embedded; IVF_PQ with refine_factor

In [ ]:
from pathlib import Path

from IPython.display import Markdown

failed = run.finish()  # writes results.md and results.json; returns the number of failed checks
Markdown(Path("/kaggle/working/results.md").read_text())